In [0]:
import requests
import json

webhook_url = dbutils.secrets.get(
    scope="prj-secret",
    key="slack-webhook"
)

def send_slack_message(message):

    headers = {
        "Content-Type": "application/json"
    }

    payload = {
        "text": message
    }

    requests.post(
        webhook_url,
        headers=headers,
        data=json.dumps(payload)
    )

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
# Step 2 - Use Catalog
spark.sql("USE CATALOG Silver_Catalog")
spark.sql("USE SCHEMA Silver_SCH")

In [0]:
# Step 3 - Read Bronze Table
energy_df = spark.table(
    "Bronze_Catalog.Bronze_SCH.Bronze_Energy_Usage"
)

In [0]:
print("Total Records :", energy_df.count())
display(energy_df.limit(10))

In [0]:
# Step 4 - Verify Data
energy_df.printSchema()




In [0]:
energy_df.show(5, False)

In [0]:
# Step 5 - Remove Duplicates

energy_df = energy_df.dropDuplicates()

print(f"Records after removing duplicates : {energy_df.count()}")

In [0]:
# Step 6 - Check Null Values

from pyspark.sql.functions import col, sum, when

null_df = energy_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in energy_df.columns
])

display(null_df)

In [0]:
# Handle Null Values

from pyspark.sql.functions import col, coalesce, lit

energy_df = energy_df \
    .withColumn("voltage_reading", coalesce(col("voltage_reading"), lit(0.0))) \
    .withColumn("current_reading", coalesce(col("current_reading"), lit(0.0))) \
    .withColumn("active_power_kw", coalesce(col("active_power_kw"), lit(0.0))) \
    .withColumn("reactive_power_kvar", coalesce(col("reactive_power_kvar"), lit(0.0))) \
    .withColumn("energy_usage_kwh", coalesce(col("energy_usage_kwh"), lit(0.0))) \
    .withColumn("frequency_hz", coalesce(col("frequency_hz"), lit(0.0))) \
    .withColumn("load_factor", coalesce(col("load_factor"), lit(0.0))) \
    .withColumn("peak_demand_kw", coalesce(col("peak_demand_kw"), lit(0.0))) \
    .withColumn("offpeak_demand_kw", coalesce(col("offpeak_demand_kw"), lit(0.0))) \
    .withColumn("daily_consumption_kwh", coalesce(col("daily_consumption_kwh"), lit(0.0)))

In [0]:
# Convert Data Types

from pyspark.sql.functions import col

energy_df = energy_df  \
.withColumn("voltage_reading", col("voltage_reading").cast("double")) \
.withColumn("current_reading", col("current_reading").cast("double")) \
.withColumn("active_power_kw", col("active_power_kw").cast("double")) \
.withColumn("reactive_power_kvar", col("reactive_power_kvar").cast("double")) \
.withColumn("energy_usage_kwh", col("energy_usage_kwh").cast("double")) \
.withColumn("frequency_hz", col("frequency_hz").cast("double")) \
.withColumn("load_factor", col("load_factor").cast("double")) \
.withColumn("peak_demand_kw", col("peak_demand_kw").cast("double")) \
.withColumn("offpeak_demand_kw", col("offpeak_demand_kw").cast("double")) \
.withColumn("daily_consumption_kwh", col("daily_consumption_kwh").cast("double"))

In [0]:
energy_df.printSchema()

In [0]:
# Step 10 - Data Validation

energy_df = energy_df.filter(
    (col("voltage_reading") >= 0) &
    (col("current_reading") >= 0) &
    (col("active_power_kw") >= 0) &
    (col("reactive_power_kvar") >= 0) &
    (col("energy_usage_kwh") >= 0) &
    (col("frequency_hz").between(45, 65)) &
    (col("load_factor").between(0, 1))
)

In [0]:
# Step 12 - Create Surrogate Key

from pyspark.sql.functions import monotonically_increasing_id

energy_df = energy_df.withColumn(
    "energy_sk",
    monotonically_increasing_id()
)

In [0]:
energy_df.select(
    "energy_sk",
    "household_id"
).show(10, False)

In [0]:
from pyspark.sql.functions import expr

energy_df = energy_df \
.withColumn("voltage_reading", expr("try_cast(voltage_reading as double)")) \
.withColumn("current_reading", expr("try_cast(current_reading as double)")) \
.withColumn("active_power_kw", expr("try_cast(active_power_kw as double)")) \
.withColumn("reactive_power_kvar", expr("try_cast(reactive_power_kvar as double)")) \
.withColumn("energy_usage_kwh", expr("try_cast(energy_usage_kwh as double)")) \
.withColumn("frequency_hz", expr("try_cast(frequency_hz as double)")) \
.withColumn("load_factor", expr("try_cast(load_factor as double)")) \
.withColumn("peak_demand_kw", expr("try_cast(peak_demand_kw as double)")) \
.withColumn("offpeak_demand_kw", expr("try_cast(offpeak_demand_kw as double)")) \
.withColumn("daily_consumption_kwh", expr("try_cast(daily_consumption_kwh as double)"))

In [0]:
energy_df.select(
    "energy_sk",
    "household_id"
).show(10, False)

In [0]:
# Step 13 - Add SCD Type 2 Columns
from pyspark.sql.functions import current_timestamp, lit

energy_df = energy_df \
    .withColumn("effective_from", current_timestamp()) \
    .withColumn("effective_to", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

In [0]:
energy_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema","true") \
    .saveAsTable("Silver_Catalog.Silver_SCH.Silver_Energy_Usage")

In [0]:
from pyspark.sql.functions import current_timestamp

records_loaded = energy_df.count()

try:

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Energy_Usage',
        'Energy Silver Pipeline',
        current_timestamp(),
        {records_loaded},
        'SUCCESS',
        NULL
    )
    """)

    print(f"✅ Silver_Energy_Usage loaded successfully. Records Loaded: {records_loaded}")

    send_slack_message(f"""
✅ Silver_Energy_Usage SUCCESS

Pipeline : Energy Silver Pipeline

Records Loaded : {records_loaded}

Status : SUCCESS
""")

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
    INSERT INTO Silver_Catalog.Silver_SCH.Audit_Log
    VALUES(
        'Silver_Energy_Usage',
        'Energy Silver Pipeline',
        current_timestamp(),
        0,
        'FAILED',
        '{error_message}'
    )
    """)
    print(f"❌ Silver_Energy_Usage failed : {error_message}")

    send_slack_message(f"""
❌ Silver_Energy_Usage FAILED

Pipeline : Energy Silver Pipeline

Error :

{error_message}
""")

    raise

In [0]:
# OPTIMIZE
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Energy_Usage
""")

In [0]:
# ZORDER
spark.sql("""
OPTIMIZE Silver_Catalog.Silver_SCH.Silver_Energy_Usage
ZORDER BY (household_id)
""")

In [0]:
# VACUUM
spark.sql("""
VACUUM Silver_Catalog.Silver_SCH.Silver_Energy_Usage
RETAIN 168 HOURS
""")

In [0]:
# Time Travel Verify
spark.sql("""
DESCRIBE HISTORY
Silver_Catalog.Silver_SCH.Silver_Energy_Usage
""").show(truncate=False)

In [0]:
energy_df.select(
    "energy_sk",
    "household_id",
    "effective_from",
    "effective_to",
    "is_current"
).show(10, False)